# SKU x Location Hierarchical Demand Forecasting — EDA

This notebook explores the real SKU x Location panel (20 SKUs x 5 stores,
760 days) and demonstrates the building blocks used by `src/pipeline.py`:
probabilistic (quantile) forecasting, bottom-up hierarchical reconciliation,
CUSUM drop detection, and calibration diagnostics.

**Data:** this project uses only the real dataset, adapted via
`python -m src.prepare_real_data --input <original_csv>`. There is no
synthetic-data fallback anywhere in this project.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from src import pipeline, data_loader, drop_detection, plotting, config

%matplotlib inline

In [ ]:
df = pipeline.load_data()
print(df.shape)
df.head()

## Data characteristics: distribution & weekly seasonality

In [ ]:
plotting.plot_demand_distribution_and_seasonality(df)
plt.show()

## Demand heatmap: SKU x Location

In [ ]:
plotting.plot_demand_heatmap(df)
plt.show()

## Demand by SKU x Location (sample series)

In [ ]:
pairs = data_loader.list_sku_location_pairs(df)
fig, ax = plt.subplots(figsize=(14, 6))
for sku, location in pairs[:6]:
    series = data_loader.get_series(df, sku, location)
    ax.plot(series.index, series.values, label=f'{sku} x {location}', alpha=0.8)
ax.legend(fontsize=8)
ax.set_title('Daily demand by SKU x Location (first 6 leaves)')
plt.show()

## CUSUM drop detection on one leaf

In [ ]:
sku, location = pairs[0]
series = data_loader.get_series(df, sku, location)
result = drop_detection.detect_drops(series)
print(f'{len(result.alarm_days)} alarm days detected')
drop_detection.summarize_alarms(result).head(10)

## Next: run the full pipeline

```bash
python scripts/run_pipeline.py
# or, for a quick smoke test on a subset:
python scripts/run_pipeline.py --max-leaves 10
```

This runs, for every SKU x Location leaf: a walk-forward backtest of
conformal-calibrated quantile XGBoost (pinball loss + newsvendor cost
derived from that SKU's price), CUSUM drop detection, bottom-up
reconciliation to SKU/Location/Total levels, and a portfolio-wide
calibration check (reliability diagram + residual diagnostics) — see
`reports/figures/` after running.